# CS781 Phase 6 — Experiment 2
## Phase-1 Model 4 (Efficient Decoder) + Standard LoRA

**Goal:** improve abstractive legal summarization using conventional additive Low-Rank Adaptation.

### Adaptation
For each selected frozen projection:

\[
W' = W_0 + \frac{\alpha}{r}BA
\]

- Rank `r = 16`
- Alpha `α = 32` (scaling = 2.0)
- Adapter dropout = 0.05
- Targets: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- Base Phase-1 weights remain frozen.

### Experimental controls
- Same Phase-6 data source, duplicate-grouped split seed, prompt, tokenizer, source/target limits and evaluator as Experiment 1.
- Three-epoch score-focused training budget; periodic validation selects and retains the best adapter checkpoint.
- The final validation score is computed on 100 held-out documents. A separate 32-document subset tunes decoding.
- Test inference is a separate final step and does not use gold summaries.

**Important:** This notebook intentionally contains no knowledge-distillation code, teacher-logit export, or teacher model.


### Changes in this revision
- **Crash fix:** the metadata cell used undefined `n_trainable` / `n_total` and would have failed after training; the 13B cell also silently overwrote `experiment_metadata.json`.
- **Seed consistency:** seed 2026 matches Experiment 1's duplicate-grouped validation split.
- **Source selection:** a sentence scorer fitted on the official training data fills the source window instead of "first 2/3 + last 1/3"; the best of lead / head+tail / tf-idf / learned is chosen by reference-summary content coverage on a separate selector-tuning slice.
- **Generation:** batched KV-cached decoding (LoRA merged into the weights), decoding setting tuned on 32 held-out validation documents (min length, no-repeat n-gram, repetition penalty). The old fixed `no_repeat_ngram=5` (about 3 words) blocks legitimate repeats such as statute names. The 100 scored validation documents are the same ones as Exp 1.
- Test generation is resumable per batch.

### Runtime budget
Designed for a Kaggle T4 run within approximately 3–4 hours, based on Experiment 1's measured throughput:
- 3 training epochs (approximately 70–90 minutes)
- 100 validation generations + BERTScore/ROUGE (approximately 45–55 minutes)
- 100 test generations + ZIP checks (approximately 45–55 minutes)

Actual runtime depends on generated summary lengths, GPU availability, and package/model downloads. The notebook prints per-stage progress and ETA.


In [1]:
# 0. Environment and imports
import os, sys, json, math, random, time, gc, hashlib, re, subprocess, importlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 2026

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"
if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # bf16 only where it is natively supported (Ampere+); T4/P100 use fp16 + GradScaler.
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float16   # unused: autocast is disabled on CPU
USE_GRAD_SCALER = USE_CUDA and AMP_DTYPE == torch.float16
torch.set_float32_matmul_precision("high")

def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_CUDA)

def ensure_package(import_name, pip_name=None):
    # Import a package, pip-installing it first if it is missing.
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])
        importlib.invalidate_caches()
        return importlib.import_module(import_name)

print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if USE_CUDA else "CPU",
      "| AMP:", str(AMP_DTYPE).replace("torch.", "") if USE_CUDA else "off")


torch 2.11.0+cu128 | device: cuda | Tesla T4 | AMP: float16


In [2]:
# 1. CONFIGURATION — Model 4 + standard LoRA
# Leave paths as None for safe auto-discovery, or set explicit Kaggle paths.

CHECKPOINT_PATH = None
TOKENIZER_PATH  = None
TRAIN_JSON_PATH = None
TEST_JSON_PATH  = None

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DATASET_SUBDIR = "Phase_6_Summ"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"
CHECKPOINT_FILENAME = "eff_full_best (6).pt"

ON_KAGGLE = Path("/kaggle/input").exists()
OUTPUT_DIR = Path("/kaggle/working/phase6_exp2_model4_lora") if ON_KAGGLE else Path("./phase6_exp2_model4_lora")
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep identical to Experiment 1 for a fair comparison.
SEED = 2026   # matches the Experiment-1 duplicate-grouped validation split
VAL_FRACTION = 0.05
MAX_SOURCE_TOKENS = 1500
MAX_TARGET_TOKENS = 800
MAX_SEQ_LEN = 2400

# Score-focused budget, bounded to the requested 3–4 hour total runtime.
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LOG_EVERY = 50
EVAL_EVERY = 500
SAVE_EVERY = 500
EVAL_MAX_EXAMPLES = 100

RESUME = False
RESUME_TEST_PREDICTIONS = True

# Standard LoRA: W' = W0 + (alpha / rank) * B(A(dropout(x)))
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj")

# Decoding and validation. Full 100-document validation is retained for a stable comparison.
GEN_MAX_NEW_TOKENS = 800
NO_REPEAT_NGRAM_SIZE = 0     # only used by the old per-document generate_summary; the tuned grid below decides
VAL_SCORE_EXAMPLES = 100
BERTSCORE_BATCH_SIZE = 8
RUN_TEST_INFERENCE = True

# ---- source selection (replaces plain head+tail) ----
SELECTOR_MODE = "auto"        # "auto" | "lead" | "headtail" | "tfidf" | "learned"
SELECTOR_FIT_DOCS = 2500      # training docs used to fit the sentence scorer
SELECTOR_EVAL_DOCS = 150      # validation docs used to compare selectors by coverage

# ---- batched generation + decoding-grid tuning ----
GEN_BATCH_SIZE = 8            # lower to 4 if CUDA runs out of memory (it also halves a batch automatically)
N_TUNE_DOCS = 32              # validation docs val_rows[100:132] (disjoint from the 100 scored docs)
# min_frac: EOS blocked until min_frac*GEN_MAX_NEW_TOKENS tokens; ngram: no-repeat n-gram (tokens; 0=off); rep: repetition penalty
GEN_GRID = [
    dict(name="greedy",           min_frac=0.00, ngram=0,  rep=1.00),
    dict(name="greedy+min",       min_frac=0.45, ngram=0,  rep=1.00),
    dict(name="ng12+min",         min_frac=0.45, ngram=12, rep=1.00),
    dict(name="ng20+min",         min_frac=0.45, ngram=20, rep=1.00),
    dict(name="ng20+rep1.05+min", min_frac=0.45, ngram=20, rep=1.05),
]
set_seed(SEED)

assert MICRO_BATCH_SIZE >= 1 and GRAD_ACCUM_STEPS >= 1 and NUM_EPOCHS >= 1
assert 0.0 < VAL_FRACTION < 0.5 and 0.0 <= WARMUP_RATIO < 1.0
assert LORA_RANK > 0 and LORA_ALPHA > 0 and 0 <= NO_REPEAT_NGRAM_SIZE
print("Output:", OUTPUT_DIR.resolve())
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, scaling={LORA_ALPHA/LORA_RANK:.3f}, dropout={LORA_DROPOUT}")
print("Targets:", LORA_TARGETS)
print(f"Training budget: {NUM_EPOCHS} epochs; full validation generation: {VAL_SCORE_EXAMPLES} docs")



Output: /kaggle/working/phase6_exp2_model4_lora
LoRA: rank=16, alpha=32, scaling=2.000, dropout=0.05
Targets: ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
Training budget: 3 epochs; full validation generation: 100 docs


In [3]:
# 2. Locate model files and fetch Phase 6 data from Hugging Face
def find_unique(filename, explicit=None):
    if explicit:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    hits = {}
    for root in SEARCH_ROOTS:
        if root.exists():
            for x in root.rglob(filename):
                if x.is_file():
                    hits[x.resolve()] = x
    hits = list(hits.values())
    if not hits:
        raise FileNotFoundError(
            f"Could not find {filename}. Set its explicit path in the configuration cell."
        )
    if len(hits) > 1:
        print("Multiple candidates for", filename)
        for x in hits[:20]:
            print("  ", x)
        raise RuntimeError(f"Set an explicit path for {filename} in the configuration cell.")
    return hits[0]

def get_hf_token():
    # Kaggle Secrets is optional for public datasets, but required for gated/private access.
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
        except Exception as exc:
            print(
                f"Could not read Kaggle secret '{HF_TOKEN_SECRET_NAME}'. "
                "Continuing without a token; this works only if the dataset is public."
            )
    return None

def download_phase6_jsons():
    try:
        from huggingface_hub import HfApi, hf_hub_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import HfApi, hf_hub_download

    token = get_hf_token()
    api = HfApi(token=token)
    repo_files = api.list_repo_files(
        repo_id=HF_DATASET_REPO, repo_type="dataset", token=token
    )
    print(f"Found {len(repo_files)} files in dataset repository {HF_DATASET_REPO}")

    def choose_file(kind):
        # Accommodate the repository's Train/Test folders and minor filename variations.
        candidates = [
            name for name in repo_files
            if name.lower().endswith(".json")
            and HF_DATASET_SUBDIR.lower() in name.lower()
            and kind.lower() in [part.lower() for part in name.replace("\\", "/").split("/")]
        ]
        # Prefer exact conventional names if multiple JSONs exist.
        preferred = [name for name in candidates
                     if Path(name).name.lower() == f"{kind.lower()}.json"]
        candidates = preferred or candidates
        if len(candidates) != 1:
            print(f"Candidate {kind} JSON files:")
            for name in candidates:
                print("  ", name)
            raise FileNotFoundError(
                f"Expected exactly one {kind} JSON under {HF_DATASET_SUBDIR}; "
                "inspect the listed candidates and set TRAIN_JSON_PATH/TEST_JSON_PATH explicitly."
            )
        return candidates[0]

    train_file = choose_file("Train")
    test_file = choose_file("Test")
    print("HF train file:", train_file)
    print("HF test file :", test_file)

    train_local = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=train_file, repo_type="dataset", token=token
    )
    test_local = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=test_file, repo_type="dataset", token=token
    )
    return Path(train_local), Path(test_local)

# The Phase 1 checkpoint and tokenizer are separate from this HF dataset.
CHECKPOINT_PATH = find_unique(CHECKPOINT_FILENAME, CHECKPOINT_PATH)
TOKENIZER_PATH = find_unique("tokenizer.json", TOKENIZER_PATH)

if TRAIN_JSON_PATH is None or TEST_JSON_PATH is None:
    hf_train_path, hf_test_path = download_phase6_jsons()
    TRAIN_JSON_PATH = Path(TRAIN_JSON_PATH) if TRAIN_JSON_PATH else hf_train_path
    TEST_JSON_PATH = Path(TEST_JSON_PATH) if TEST_JSON_PATH else hf_test_path
else:
    TRAIN_JSON_PATH = find_unique("train.json", TRAIN_JSON_PATH)
    TEST_JSON_PATH = find_unique("test.json", TEST_JSON_PATH)

for label, pth in [("Checkpoint", CHECKPOINT_PATH), ("Tokenizer ", TOKENIZER_PATH),
                   ("Train     ", TRAIN_JSON_PATH), ("Test      ", TEST_JSON_PATH)]:
    if not Path(pth).is_file():
        raise FileNotFoundError(f"{label.strip()} file does not exist: {pth}")
    print(f"{label}: {pth}")


Found 34 files in dataset repository Exploration-Lab/CS-781-Capstone
HF train file: Phase_6_Summ/Train/train.json
HF test file : Phase_6_Summ/Test/test.json


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Tokenizer : /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Train     : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json
Test      : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Test/test.json


In [4]:
# 3. Phase 1 tokenizer (byte-level BPE; must behave exactly like the Phase 1 implementation)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
_PRETOKEN_RE = re.compile(r"\S+|\s+")   # words and whitespace runs are BPE'd independently

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    pad_token_id = property(lambda self: self.stoi["<pad>"])
    unk_token_id = property(lambda self: self.stoi["<unk>"])
    bos_token_id = property(lambda self: self.stoi["<s>"])
    eos_token_id = property(lambda self: self.stoi["</s>"])

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l, r): i for i, (l, r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    @classmethod
    def load_json(cls, path):
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)
        tok.symbol_bytes = {f"{b:02x}": bytes([b]) for b in range(256)}
        for l, r in tok.merges:
            tok.symbol_bytes[l + r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]
        tok._reset_encode_cache()
        return tok

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()
        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)
        symbols = self.word_to_symbols(word)
        ranks = self._ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for j in range(len(symbols) - 1):
                r = ranks.get((symbols[j], symbols[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (symbols[j], symbols[j + 1]), r
            if best is None:
                break
            left, right = best
            merged, j = [], 0
            while j < len(symbols):
                if j + 1 < len(symbols) and symbols[j] == left and symbols[j + 1] == right:
                    merged.append(left + right); j += 2
                else:
                    merged.append(symbols[j]); j += 1
            symbols = merged
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        unk = self.unk_token_id
        out = []
        for unit in _PRETOKEN_RE.findall(text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok not in BPE_SPECIALS:
                buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
PAD_ID, BOS_ID, EOS_ID = tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id
print("Tokenizer vocab:", len(tokenizer))

# Round-trip must be lossless for plain text, newlines, tabs, and non-ASCII characters.
for probe in ["IN THE SUPREME COURT OF INDIA — Section 14.",
              "Line one.\n\nLine two\twith tab,  double  space.",
              "Appeal No. 123/2019 (धारा 302) — decided on 5th May, 2019."]:
    assert tokenizer.decode(tokenizer.encode(probe)) == probe, f"round-trip failed: {probe!r}"
print("Tokenizer round-trip OK")

Tokenizer vocab: 24576
Tokenizer round-trip OK


In [5]:
# 4. Model 4 architecture (GQA + RoPE + RMSNorm + SwiGLU). Parameter names match the Phase 1 checkpoint;
#    a KV-cache path is added for fast generation (it does not change any parameter or the maths).
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return x32.to(dtype) * self.weight

class RotaryTable(nn.Module):
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
    def forward(self, position_ids):
        return self.cos_cached[position_ids].unsqueeze(1), self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x, cos, sin):
    B, H, T, D = x.shape
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k, v = torch.cat([pk, k], 2), torch.cat([pv, v], 2)
        new_kv = (k, v) if use_cache else None
        S = k.size(2)
        if self.kv_repeat != 1:
            ka = k.repeat_interleave(self.kv_repeat, dim=1)
            va = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            ka, va = k, v
        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]
        y = F.scaled_dot_product_attention(q, ka, va, attn_mask=attn_mask,
                                          dropout_p=self.dropout if self.training else 0.0,
                                          is_causal=is_causal)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past_kv=None, use_cache=False):
        a, new_kv = self.attn(self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache)
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv

class EfficientDecoderLM(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = cfg["d_model"]
        self.max_positions = cfg["max_positions"]
        dropout = cfg.get("dropout", 0.0)
        self.token_emb = nn.Embedding(vocab_size, self.d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(self.d_model // cfg["n_heads"], self.max_positions, cfg["rope_base"])
        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(self.d_model, cfg["n_heads"], cfg["n_kv_heads"],
                                  cfg["ffn_dim"], dropout, cfg["rms_eps"])
            for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(self.d_model, cfg["rms_eps"])

    def forward_hidden(self, input_ids, past_kvs=None, use_cache=False):
        # Returns hidden states, or (hidden, new_kvs) when use_cache=True.
        B, T = input_ids.shape
        past_len = 0 if past_kvs is None else past_kvs[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"sequence {past_len + T} > max_positions {self.max_positions}")
        pos = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(pos)
        x = self.drop(self.token_emb(input_ids))
        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, kv = block(x, cos, sin, None if past_kvs is None else past_kvs[i], use_cache)
            new_kvs.append(kv)
        h = self.final_norm(x)
        return (h, new_kvs) if use_cache else h

    def forward(self, input_ids):
        return F.linear(self.forward_hidden(input_ids), self.token_emb.weight)   # tied embeddings

In [6]:
# 5. Load the Phase 1 checkpoint and verify architecture / weight compatibility
def load_torch(path):
    # Prefer the safe tensor-only loader; fall back to full unpickling only for files you trust.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception as e:
        print(f"weights_only load failed ({type(e).__name__}); falling back to full unpickling for {Path(path).name}")
        return torch.load(path, map_location="cpu", weights_only=False)

ckpt = load_torch(CHECKPOINT_PATH)
if "model" not in ckpt or "config" not in ckpt:
    raise ValueError("Expected a Model 4 checkpoint with 'model' and 'config' entries.")
cfg = dict(ckpt["config"])

# Phase 1 checkpoints may omit the RMSNorm epsilon or store it under another key.
# Normalize common aliases first; if none is present, use the RMSNorm class default
# (1e-5) used by this reconstructed Model 4 implementation.
if "rms_eps" not in cfg:
    for eps_key in ("rms_norm_eps", "norm_eps", "layer_norm_epsilon", "eps"):
        if eps_key in cfg:
            cfg["rms_eps"] = float(cfg[eps_key])
            print(f"Using checkpoint RMSNorm epsilon from config['{eps_key}'] = {cfg['rms_eps']}")
            break
    else:
        cfg["rms_eps"] = 1e-5
        print(
            "WARNING: checkpoint config has no RMSNorm epsilon field. "
            "Using rms_eps=1e-5 (the RMSNorm implementation default). "
            "If your original Phase 1 Model 4 used a different epsilon, set cfg['rms_eps'] "
            "to that exact value before constructing the model."
        )

REQUIRED_CFG = ["d_model", "n_layers", "n_heads", "n_kv_heads", "ffn_dim",
                "max_positions", "rope_base", "rms_eps"]
missing_cfg = [k for k in REQUIRED_CFG if k not in cfg]
if missing_cfg:
    raise KeyError(f"Checkpoint config missing: {missing_cfg}")
if int(cfg.get("vocab_size", len(tokenizer))) != len(tokenizer):
    raise ValueError(f"Tokenizer/checkpoint vocab mismatch: {len(tokenizer)} vs {cfg.get('vocab_size')}")
if MAX_SEQ_LEN > int(cfg["max_positions"]):
    raise ValueError(f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds checkpoint max_positions={cfg['max_positions']}")

model = EfficientDecoderLM(cfg, len(tokenizer))
model.load_state_dict(ckpt["model"], strict=True)     # raises on any missing/unexpected key
model.to(DEVICE).eval()
del ckpt; gc.collect()
print(f"Loaded Model 4: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print("Architecture:", {k: cfg[k] for k in REQUIRED_CFG}, "| dropout:", cfg.get("dropout", 0.0))

Loaded Model 4: 99.3M parameters
Architecture: {'d_model': 640, 'n_layers': 17, 'n_heads': 10, 'n_kv_heads': 2, 'ffn_dim': 2048, 'max_positions': 2560, 'rope_base': 10000.0, 'rms_eps': 1e-05} | dropout: 0.0


In [7]:
# 6. Attach standard LoRA adapters to all selected Model-4 projections
# Standard LoRA: y = W0(x) + (alpha / r) * B(A(dropout(x))).
# A is Kaiming-initialised and B is zero-initialised, so the initial adapted
# network is exactly function-preserving relative to the frozen Phase-1 model.

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError(f"LoRA expected nn.Linear, got {type(base)}")
        if rank <= 0:
            raise ValueError("LoRA rank must be positive")
        self.base = base
        self.base.requires_grad_(False)
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.lora_dropout = nn.Dropout(float(dropout))
        self.lora_A = nn.Parameter(torch.empty(self.rank, base.in_features,
                                               device=base.weight.device, dtype=base.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, self.rank,
                                               device=base.weight.device, dtype=base.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base(x)
        low_rank = F.linear(F.linear(self.lora_dropout(x), self.lora_A), self.lora_B)
        return base_out + self.scaling * low_rank


def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False


def inject_lora(model, targets=LORA_TARGETS, rank=LORA_RANK,
                alpha=LORA_ALPHA, dropout=LORA_DROPOUT):
    freeze_all(model)
    replaced = []
    for i, block in enumerate(model.blocks):
        groups = (
            ("attn", block.attn, ("q_proj", "k_proj", "v_proj", "o_proj")),
            ("ff", block.ff, ("gate_proj", "up_proj", "down_proj")),
        )
        for group_name, module, names in groups:
            for name in names:
                if name in targets:
                    old = getattr(module, name)
                    setattr(module, name, LoRALinear(old, rank, alpha, dropout))
                    replaced.append(f"blocks.{i}.{group_name}.{name}")
    return replaced


def is_lora_key(key):
    return key.endswith(".lora_A") or key.endswith(".lora_B")


def lora_state_dict(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()
            if is_lora_key(k)}


def load_lora_state_dict(module, state):
    expected = {k for k in module.state_dict() if is_lora_key(k)}
    if set(state) != expected:
        raise RuntimeError(
            f"LoRA adapter keys mismatch: missing={sorted(expected-set(state))[:4]}, "
            f"unexpected={sorted(set(state)-expected)[:4]}"
        )
    module.load_state_dict(state, strict=False)


# Verify the original model function is unchanged at adapter initialisation.
model.eval()
probe_ids = (torch.arange(48, device=DEVICE) % (len(tokenizer) - 4) + 4)[None, :]
with torch.inference_mode():
    ref_logits = model(probe_ids).float()
replaced = inject_lora(model)
model.eval()
with torch.inference_mode():
    adapted_logits = model(probe_ids).float()
identity_diff = (ref_logits - adapted_logits).abs().max().item()
assert identity_diff < 1e-5, f"LoRA init changed model output: max diff={identity_diff:.3e}"

expected_modules = int(cfg["n_layers"]) * len(LORA_TARGETS)
assert len(replaced) == expected_modules, (len(replaced), expected_modules)
lora_params = [(n, p) for n, p in model.named_parameters()
               if p.requires_grad and (".lora_A" in n or ".lora_B" in n)]
assert len(lora_params) == 2 * expected_modules
assert all(torch.count_nonzero(p.detach()).item() == 0
           for n, p in lora_params if ".lora_B" in n)

trainable = sum(p.numel() for _, p in lora_params)
total = sum(p.numel() for p in model.parameters())
assert all(not p.requires_grad for n, p in model.named_parameters()
           if ".base." in n or n.startswith(("token_emb.", "final_norm.", "blocks.")) and
           ".lora_A" not in n and ".lora_B" not in n)

print("Injected LoRA modules:", len(replaced), "/", expected_modules)
print("Trainable LoRA parameters:", f"{trainable:,}", f"({100*trainable/total:.3f}% of model)")
print("Frozen model parameters:", f"{total-trainable:,}")
print("LoRA scaling alpha/r:", LORA_ALPHA / LORA_RANK)
print("Identity-at-initialisation max-logit difference:", f"{identity_diff:.3e}")
print("Projection shapes:",
      "Q", tuple(model.blocks[0].attn.q_proj.base.weight.shape),
      "K", tuple(model.blocks[0].attn.k_proj.base.weight.shape),
      "V", tuple(model.blocks[0].attn.v_proj.base.weight.shape),
      "O", tuple(model.blocks[0].attn.o_proj.base.weight.shape))


Injected LoRA modules: 119 / 119
Trainable LoRA parameters: 3,307,520 (3.223% of model)
Frozen model parameters: 99,309,440
LoRA scaling alpha/r: 2.0
Identity-at-initialisation max-logit difference: 0.000e+00
Projection shapes: Q (640, 640) K (128, 640) V (128, 640) O (640, 640)


In [8]:
# 7. Load Phase 6 data and build a duplicate-grouped train/validation split
def load_records(path):
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):                      # id-keyed JSON
        return [{"id": str(k), **v} for k, v in obj.items() if isinstance(v, dict)]
    if not isinstance(obj, list):
        raise TypeError("Expected a JSON list or an ID-keyed object")
    return obj

_WS_RE = re.compile(r"\s+")
def doc_group_key(text):
    # Normalised-document hash: exact duplicate documents always land in the same split.
    return hashlib.sha1(_WS_RE.sub(" ", text).strip().lower().encode("utf-8")).hexdigest()

train_records = load_records(TRAIN_JSON_PATH)
test_records = load_records(TEST_JSON_PATH)

clean = []
for r in train_records:
    rid, doc, summ = str(r.get("id", "")), r.get("document", ""), r.get("summary", "")
    if rid and isinstance(doc, str) and doc.strip() and isinstance(summ, str) and summ.strip():
        clean.append({"id": rid, "document": doc.strip(), "summary": summ.strip()})
for r in clean:
    r["group"] = doc_group_key(r["document"])
print(f"Raw train: {len(train_records)} | usable: {len(clean)} | dropped: {len(train_records) - len(clean)} | test: {len(test_records)}")
assert len({r["id"] for r in clean}) == len(clean), "duplicate ids in train.json"

# Validate the test file up-front so inference cannot fail halfway through.
for i, r in enumerate(test_records):
    if not str(r.get("id", "")) or not isinstance(r.get("document"), str):
        raise ValueError(f"Invalid test record at index {i}")
assert len({str(r["id"]) for r in test_records}) == len(test_records), "duplicate ids in test.json"

groups = defaultdict(list)
for r in clean:
    groups[r["group"]].append(r)
group_keys = list(groups)
random.Random(SEED).shuffle(group_keys)
target_val = max(1, int(len(clean) * VAL_FRACTION))
val_keys, n_val = set(), 0
for key in group_keys:
    if n_val >= target_val:
        break
    val_keys.add(key); n_val += len(groups[key])

train_rows = [r for r in clean if r["group"] not in val_keys]
val_rows = [r for r in clean if r["group"] in val_keys]
random.Random(SEED + 1).shuffle(val_rows)     # fixed random order -> "first N" is a random sample
assert train_rows and val_rows
assert {r["id"] for r in train_rows}.isdisjoint(r["id"] for r in val_rows)
assert {r["group"] for r in train_rows}.isdisjoint(r["group"] for r in val_rows)

train_groups = {r["group"] for r in train_rows}
test_overlap = sum(doc_group_key(r["document"].strip()) in train_groups for r in test_records)
print(f"Train={len(train_rows)} | validation={len(val_rows)} | unique documents={len(groups)} "
      f"| duplicated-doc rows={len(clean) - len(groups)}")
print(f"Test documents that also appear (exactly) in train: {test_overlap}")

Raw train: 7030 | usable: 7028 | dropped: 2 | test: 100
Train=6677 | validation=351 | unique documents=7001 | duplicated-doc rows=27
Test documents that also appear (exactly) in train: 0


In [9]:
# 7b. Sentence-level source selection (fitted on the official training data only)
from collections import Counter
from tqdm.auto import tqdm
SOURCE_BUDGET = MAX_SOURCE_TOKENS
SPACE_IDS = tokenizer.encode(" ")

STOP = set(("a an the and or of to in on at by for with from as is are was were be been being "
            "this that these those it its he she they them his her their which who whom whose what "
            "when where why how not nor but if then than so such there here also any all each other "
            "into upon under over after before between against during without within through can "
            "could may might must shall should will would do does did has have had we you our us your").split())
WORD_RE = re.compile(r"[a-z]{3,}")
def content_words(text): return [w for w in WORD_RE.findall(text.lower()) if w not in STOP]

_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9\"'(\[])")
_ABBR = re.compile(r"(?:\b[A-Z]|\b(?:No|Nos|Sec|Secs|Art|Mr|Mrs|Ms|Dr|St|Rs|Ltd|Co|Cr|Pvt|Smt|Shri|Hon|Prof|vs|etc|viz|Ex|Dy|Addl|Govt|Corpn|Bros|para|paras|cl|ss|Pr))\.$")

def split_sentences(doc, max_words=80, chunk_words=50):
    parts = _SPLIT.split(doc)
    merged, buf = [], ""
    for p in parts:
        buf = (buf + " " + p) if buf else p
        if _ABBR.search(buf) and len(buf.split()) < 60:
            continue
        merged.append(buf); buf = ""
    if buf: merged.append(buf)
    out = []
    for s in merged:
        s = s.strip()
        if not s: continue
        w = s.split()
        if len(w) > max_words:
            for j in range(0, len(w), chunk_words): out.append(" ".join(w[j:j + chunk_words]))
        else: out.append(s)
    return out

CUES = ["appellant", "respondent", "petitioner", "held", "court", "section", "act", "contended", "submitted",
        "argued", "dismissed", "allowed", "appeal", "tribunal", "order", "judgment", "therefore", "accordingly",
        "question", "issue", "facts", "filed", "hence", "opinion", "decision"]
FEAT_NAMES = ["relpos", "from_start", "from_end", "log_nsent", "log_nwords", "n_content",
              "digit_ratio", "cap_ratio", "tfidf_mean", "tf_mean"] + CUES
TFIDF_COL = FEAT_NAMES.index("tfidf_mean")
IDF, IDF_DEFAULT, GBM = None, None, None

def build_idf(rows):
    df = Counter()
    for r in rows: df.update(set(content_words(r["document"])))
    N = len(rows)
    return {w: math.log((N + 1) / (c + 1)) + 1.0 for w, c in df.items()}, math.log(N + 1) + 1.0

def sentence_features(doc, idf=None, idf_default=None):
    idf = IDF if idf is None else idf
    idf_default = IDF_DEFAULT if idf_default is None else idf_default
    sents = split_sentences(" ".join(doc.split())); n = len(sents)
    toks = [WORD_RE.findall(s.lower()) for s in sents]
    cws = [[w for w in t if w not in STOP] for t in toks]
    tf = Counter(w for c in cws for w in c)
    X = np.zeros((n, len(FEAT_NAMES)), dtype=np.float32)
    for i, (s, t, c) in enumerate(zip(sents, toks, cws)):
        words = s.split(); nw = max(1, len(words)); nc = len(c)
        if nc:
            tfidf = sum(math.log1p(tf[w]) * idf.get(w, idf_default) for w in c) / nc
            tfm = sum(tf[w] for w in c) / nc
        else: tfidf = tfm = 0.0
        X[i, 0] = i / max(1, n - 1); X[i, 1] = min(i, 60) / 60.0; X[i, 2] = min(n - 1 - i, 60) / 60.0
        X[i, 3] = math.log1p(n); X[i, 4] = math.log1p(nw); X[i, 5] = nc
        X[i, 6] = sum(1 for w in words if any(ch.isdigit() for ch in w)) / nw
        X[i, 7] = sum(1 for w in words if w[:1].isupper()) / nw
        X[i, 8] = tfidf; X[i, 9] = tfm
        ts = set(t)
        for j, cw in enumerate(CUES):
            if cw in ts: X[i, 10 + j] = 1.0
    return sents, X, cws

def sentence_labels(cws, summary, idf, idf_default):
    S = set(content_words(summary))
    y = np.zeros(len(cws), dtype=np.float32); w = np.zeros(len(cws), dtype=np.float32)
    for i, c in enumerate(cws):
        u = set(c)
        if len(u) < 3: continue
        den = sum(idf.get(x, idf_default) for x in u)
        num = sum(idf.get(x, idf_default) for x in u if x in S)
        y[i] = num / den; w[i] = min(len(u), 20)
    return y, w

def fit_selector(rows):
    from sklearn.ensemble import HistGradientBoostingRegressor
    idf, d = build_idf(rows)
    Xs, ys, ws = [], [], []
    for r in tqdm(rows, desc="selector features"):
        sents, X, cws = sentence_features(r["document"], idf, d)
        y, w = sentence_labels(cws, r["summary"], idf, d)
        m = w > 0
        Xs.append(X[m]); ys.append(y[m]); ws.append(w[m])
    X = np.concatenate(Xs); y = np.concatenate(ys); w = np.concatenate(ws)
    print(f"fitting sentence scorer on {len(y):,} sentences ...")
    gbm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, max_leaf_nodes=31,
                                        min_samples_leaf=40, random_state=SEED)
    gbm.fit(X, y, sample_weight=w)
    return gbm, idf, d

def headtail_order(n):
    head, tail, out = 0, n - 1, []
    while head <= tail:
        for _ in range(2):
            if head <= tail: out.append(head); head += 1
        if head <= tail: out.append(tail); tail -= 1
    return out

def priority_order(mode, X):
    n = X.shape[0]
    if mode == "lead": return list(range(n))
    if mode == "headtail": return headtail_order(n)
    if mode == "tfidf": return [int(i) for i in np.argsort(-X[:, TFIDF_COL], kind="stable")]
    if mode == "learned": return [int(i) for i in np.argsort(-GBM.predict(X), kind="stable")]
    raise ValueError(mode)

def pack_sentences(sents, order, budget):
    # Add sentences by priority while they fit; return token ids in ORIGINAL document order.
    sp = len(SPACE_IDS); chosen = {}; used = 0; fails = 0
    for i in order:
        ids = tokenizer.encode(sents[i])
        need = len(ids) + (sp if chosen else 0)
        if need <= budget - used:
            chosen[i] = ids; used += need; fails = 0
        else:
            fails += 1
            if fails >= 40: break
        if budget - used < 4: break
    out = []
    for k, i in enumerate(sorted(chosen)):
        if k: out.extend(SPACE_IDS)
        out.extend(chosen[i])
    return out, sorted(chosen)

def select_from_features(sents, X, mode):
    if len(sents) == 0: return [], []
    return pack_sentences(sents, priority_order(mode, X), SOURCE_BUDGET)

def select_source_ids(doc, mode=None):
    # Returns (token ids <= MAX_SOURCE_TOKENS, True if part of the document was left out).
    sents, X, cws = sentence_features(doc)
    ids, sel = select_from_features(sents, X, mode or SELECTOR_MODE)
    if not ids:
        ids = tokenizer.encode(" ".join(doc.split()))[:SOURCE_BUDGET]
        return ids, True
    return ids, len(sel) < len(sents)

def coverage(sel, cws, summary):
    # IDF-weighted recall of the reference summary's content words inside the selected source.
    S = set(content_words(summary))
    if not S: return None
    got = set()
    for i in sel: got.update(cws[i])
    num = sum(IDF.get(w, IDF_DEFAULT) for w in S if w in got)
    den = sum(IDF.get(w, IDF_DEFAULT) for w in S)
    return num / den

In [10]:
# 7c. Fit the sentence scorer and pick the source selector by coverage of reference-summary content words
fit_rows = random.Random(SEED).sample(train_rows, min(SELECTOR_FIT_DOCS, len(train_rows)))
print(f"Fitting sentence selector using {len(fit_rows)} training documents only; "
      f"validation summaries are not used to fit the selector.")
t_selector_fit = time.time()
GBM, IDF, IDF_DEFAULT = fit_selector(fit_rows)
print(f"Selector fitting took {(time.time() - t_selector_fit)/60:.1f} min")

_modes = ["lead", "headtail", "tfidf", "learned"]
# Keep selector tuning disjoint from both the 32 decoding-tuning documents
# (val_rows[100:132]) and the 100 final scored validation documents (val_rows[:100]).
SELECTOR_EVAL_START = VAL_SCORE_EXAMPLES + N_TUNE_DOCS
SELECTOR_EVAL_END = SELECTOR_EVAL_START + SELECTOR_EVAL_DOCS
selector_eval_rows = val_rows[SELECTOR_EVAL_START:SELECTOR_EVAL_END]
if len(selector_eval_rows) < SELECTOR_EVAL_DOCS:
    raise ValueError(
        f"Need at least {SELECTOR_EVAL_END} validation rows to keep selector tuning, "
        f"decoding tuning and final scoring disjoint; found {len(val_rows)}. "
        "Reduce SELECTOR_EVAL_DOCS or VAL_FRACTION only after checking the split."
    )

cov = {m: [] for m in _modes}; cov["full_document"] = []
t_selector_eval = time.time()
for r in tqdm(selector_eval_rows, desc="selector tuning (disjoint validation slice)"):
    sents, X, cws = sentence_features(r["document"])
    if not sents: continue
    c_full = coverage(range(len(sents)), cws, r["summary"])
    if c_full is None: continue
    cov["full_document"].append(c_full)
    for m in _modes:
        ids, sel = select_from_features(sents, X, m)
        cov[m].append(coverage(sel, cws, r["summary"]))
print(f"Selector-tuning docs: {len(selector_eval_rows)} "
      f"(validation indices {SELECTOR_EVAL_START}:{SELECTOR_EVAL_END}; disjoint from final score and decoding tuning)")
print(f"Selector coverage evaluation took {(time.time() - t_selector_eval)/60:.1f} min")
print(f"Mean IDF-weighted coverage of reference-summary content words inside the {SOURCE_BUDGET}-token source window:")
for m in _modes + ["full_document"]:
    print(f"  {m:14s} {np.mean(cov[m]):.4f}")
if SELECTOR_MODE == "auto":
    SELECTOR_MODE = max(_modes, key=lambda m: np.mean(cov[m]))
print("Using source selector:", SELECTOR_MODE)

Fitting sentence selector using 2500 training documents only; validation summaries are not used to fit the selector.


selector features:   0%|          | 0/2500 [00:00<?, ?it/s]

fitting sentence scorer on 340,848 sentences ...
Selector fitting took 0.7 min


selector tuning (disjoint validation slice):   0%|          | 0/150 [00:00<?, ?it/s]

Selector-tuning docs: 150 (validation indices 132:282; disjoint from final score and decoding tuning)
Selector coverage evaluation took 0.1 min
Mean IDF-weighted coverage of reference-summary content words inside the 1500-token source window:
  lead           0.3806
  headtail       0.4076
  tfidf          0.4024
  learned        0.4144
  full_document  0.8676
Using source selector: learned


In [11]:
# 8. Build causal-LM examples:  <s> + instruction + source window + "SUMMARY:" separator + summary + </s>
# Only the summary (and EOS) is supervised. The prompt/separator are never truncated: the *source*
# window shrinks first when prompt + source + target would exceed MAX_SEQ_LEN.
PROMPT_TEMPLATE = "Summarize the following Indian court judgment accurately. Preserve the key facts, legal issues, reasoning, decision, and important citations. Do not invent facts.\n\nJUDGMENT:\n"
PROMPT_IDS = tokenizer.encode(PROMPT_TEMPLATE)
SEP_IDS = tokenizer.encode("\n\nSUMMARY:\n")
PREFIX_OVERHEAD = 1 + len(PROMPT_IDS) + len(SEP_IDS)       # BOS + instruction + separator
min_source_budget = MAX_SEQ_LEN - PREFIX_OVERHEAD - MAX_TARGET_TOKENS - 1
if min_source_budget < 256:
    raise ValueError(f"MAX_SEQ_LEN too small: only {min_source_budget} source tokens left for the longest target.")

def head_tail(ids, budget):
    # Keep the opening 2/3 and the closing 1/3 of the budget (facts early, decision late).
    # Nested property: head_tail(head_tail(x, B1), B2) == head_tail(x, B2) whenever B2 <= B1.
    assert budget >= 1
    if len(ids) <= budget:
        return ids
    left = budget * 2 // 3
    right = budget - left
    return ids[:left] + ids[-right:]

def build_prompt_ids(src_ids, budget):
    return [BOS_ID] + PROMPT_IDS + head_tail(src_ids, budget) + SEP_IDS

def tokenize_rows(rows, name):
    examples, n_trunc, t0 = [], 0, time.time()
    for i, r in enumerate(rows):
        src, was_cut = select_source_ids(r["document"])
        n_trunc += int(was_cut)
        tgt = tokenizer.encode(r["summary"])[:MAX_TARGET_TOKENS]
        budget = min(MAX_SOURCE_TOKENS, MAX_SEQ_LEN - PREFIX_OVERHEAD - len(tgt) - 1)
        prefix = build_prompt_ids(src, budget)
        ids = prefix + tgt + [EOS_ID]
        assert len(ids) <= MAX_SEQ_LEN
        examples.append((np.asarray(ids, dtype=np.int32), len(prefix)))
        if (i + 1) % 500 == 0:
            print(f"  [{name}] tokenized {i + 1}/{len(rows)} ({time.time() - t0:.0f}s)", flush=True)
    print(f"{name}: {len(examples)} examples | source truncated: {n_trunc} ({100 * n_trunc / max(1, len(rows)):.1f}%)")
    return examples

print("Tokenizing (pure-Python BPE; this can take a few minutes) ...")
train_examples = tokenize_rows(train_rows, "train")
val_examples = tokenize_rows(val_rows, "val")

class SummarizationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        ids, prefix_len = self.examples[i]
        ids_t = torch.from_numpy(ids.astype(np.int64))
        labels = ids_t.clone()
        labels[:prefix_len] = -100
        return {"input_ids": ids_t, "labels": labels}

def collate(batch):
    L = max(x["input_ids"].numel() for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        n = x["input_ids"].numel()
        ids[i, :n] = x["input_ids"]; labels[i, :n] = x["labels"]
    return {"input_ids": ids, "labels": labels}

train_ds = SummarizationDataset(train_examples)
val_ds = SummarizationDataset(val_examples)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate, num_workers=0)

lens = np.array([len(e[0]) for e in train_examples])
print(f"Train sequence length: mean={lens.mean():.0f} p95={np.percentile(lens, 95):.0f} max={lens.max()}")

Tokenizing (pure-Python BPE; this can take a few minutes) ...
  [train] tokenized 500/6677 (9s)
  [train] tokenized 1000/6677 (18s)
  [train] tokenized 1500/6677 (26s)
  [train] tokenized 2000/6677 (35s)
  [train] tokenized 2500/6677 (43s)
  [train] tokenized 3000/6677 (52s)
  [train] tokenized 3500/6677 (61s)
  [train] tokenized 4000/6677 (69s)
  [train] tokenized 4500/6677 (78s)
  [train] tokenized 5000/6677 (86s)
  [train] tokenized 5500/6677 (95s)
  [train] tokenized 6000/6677 (103s)
  [train] tokenized 6500/6677 (112s)
train: 6677 examples | source truncated: 6552 (98.1%)
val: 351 examples | source truncated: 343 (97.7%)
Train sequence length: mean=2306 p95=2362 max=2362


In [12]:
# 9. Loss helper and sanity checks
def lm_loss(model, input_ids, labels, reduction="mean"):
    # Next-token cross-entropy computed ONLY at supervised positions. The LM head is applied to
    # those positions alone (the ~1.5k prompt positions are skipped), which saves memory and time
    # and gives exactly the same value as masking with ignore_index.
    h = model.forward_hidden(input_ids)[:, :-1]
    tgt = labels[:, 1:]
    mask = tgt != -100
    n = int(mask.sum())
    if n == 0:
        return h.sum() * 0.0, 0
    logits = F.linear(h[mask], model.token_emb.weight)
    return F.cross_entropy(logits.float(), tgt[mask], reduction=reduction), n

# (a) the example layout is what we think it is
ids, plen = train_examples[0]
assert ids[0] == BOS_ID and ids[-1] == EOS_ID
assert list(ids[plen - len(SEP_IDS):plen]) == SEP_IDS, "separator missing before the target"
print("Prompt tail :", repr(tokenizer.decode(ids[max(0, plen - 60):plen])))
print("Target start:", repr(tokenizer.decode(ids[plen:plen + 60])))

# (b) the masked-head loss equals the standard full-logits ignore_index loss
model.eval()
b = collate([train_ds[0]])
x, y = b["input_ids"].to(DEVICE), b["labels"].to(DEVICE)
with torch.no_grad():
    fast, n_tok = lm_loss(model, x, y)
    full_logits = model(x)
    ref = F.cross_entropy(full_logits[:, :-1].reshape(-1, full_logits.size(-1)).float(),
                          y[:, 1:].reshape(-1), ignore_index=-100)
assert abs(float(fast) - float(ref)) < 1e-4, (float(fast), float(ref))
print(f"Loss check OK: {float(fast):.4f} (supervised tokens: {n_tok}, sequence length: {x.shape[1]})")

# (c) KV-cached decoding reproduces the full forward pass
with torch.no_grad():
    n_chk = min(200, x.shape[1])
    t = x[:, :n_chk]
    full_h = model.forward_hidden(t)
    h1, past = model.forward_hidden(t[:, :n_chk - 1], use_cache=True)
    h2, _ = model.forward_hidden(t[:, n_chk - 1:], past_kvs=past, use_cache=True)
cache_diff = (h2[:, -1] - full_h[:, -1]).abs().max().item()
assert cache_diff < 1e-3, f"KV-cache mismatch: {cache_diff}"
print(f"KV-cache check OK (max|diff|={cache_diff:.2e})")
del full_logits, full_h, h1, h2, past

Prompt tail : "sale or purchase for the purpose of consumption outside the State. The ratio of Mohd. Serajuddin vs State of Orissa, is also against the appellant 's stand.\n\nSUMMARY:\n"
Target start: 'The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell'
Loss check OK: 1.5327 (supervised tokens: 801, sequence length: 2359)
KV-cache check OK (max|diff|=3.81e-06)


In [13]:
# 10. Train LoRA adapters only (resumable; checkpoints store adapters only, not the 99M base weights)
train_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(train_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

n_micro = math.ceil(len(train_ds) / MICRO_BATCH_SIZE)           # micro-batches per epoch
steps_per_epoch = math.ceil(n_micro / GRAD_ACCUM_STEPS)         # optimizer steps per epoch
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-4, (step + 1) / warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)

log_path = OUTPUT_DIR / "train_log.jsonl"
best_path = OUTPUT_DIR / "exp2_lora_best.pt"
latest_path = OUTPUT_DIR / "exp2_lora_latest.pt"

# Avoid silently loading adapters from an earlier run when RESUME=False.
if not RESUME:
    for stale in (best_path, latest_path, log_path):
        if stale.exists():
            stale.unlink()
elif not latest_path.exists():
    raise FileNotFoundError(
        "RESUME=True but the latest training checkpoint is missing. "
        "Set RESUME=False to start a clean run."
    )
LORA_META = {"rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
              "targets": list(LORA_TARGETS), "variant": "standard LoRA"}

def atomic_save(obj, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def log_event(rec):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")

@torch.no_grad()
def evaluate_loss(max_examples=EVAL_MAX_EXAMPLES):
    # Token-weighted mean validation loss over a fixed subset.
    was_training = model.training
    model.eval()
    total, count = 0.0, 0
    for j, b in enumerate(val_loader):
        if j >= max_examples:
            break
        x, y = b["input_ids"].to(DEVICE), b["labels"].to(DEVICE)
        with amp_ctx():
            loss_sum, n = lm_loss(model, x, y, reduction="sum")
        if n and torch.isfinite(loss_sum):
            total += float(loss_sum); count += n
    model.train(was_training)
    return total / count if count else float("inf")

global_step, best_val, baseline_val = 0, float("inf"), None
if RESUME and latest_path.exists():
    st = load_torch(latest_path)
    if st["lora"] != LORA_META:
        raise RuntimeError(f"Resume checkpoint used different LoRA settings: {st['lora']} vs {LORA_META}")
    load_lora_state_dict(model, st["adapters"])
    optimizer.load_state_dict(st["optimizer"]); scheduler.load_state_dict(st["scheduler"])
    scaler.load_state_dict(st["scaler"])
    global_step, best_val, baseline_val = st["global_step"], st["best_val"], st.get("baseline_val")
    print(f"Resumed from step {global_step} (best_val={best_val:.4f})")
    del st
elif RESUME:
    print("RESUME=True but no latest checkpoint found; starting from scratch.")

if global_step == 0 and baseline_val is None:
    baseline_val = evaluate_loss()      # adapters are an exact identity here => this is the Phase 1 model
    print(f"Baseline validation loss (frozen Phase 1 model): {baseline_val:.4f}")
    log_event({"event": "baseline", "val_loss": baseline_val})

def checkpoint(val_loss=None, save_latest=True):
    global best_val
    if val_loss is not None and val_loss < best_val:
        best_val = val_loss
        atomic_save({"adapters": lora_state_dict(model), "config": cfg, "lora": LORA_META,
                     "global_step": global_step, "val_loss": val_loss,
                     "base_checkpoint": str(CHECKPOINT_PATH)}, best_path)
        print(f"  saved BEST (val_loss={best_val:.4f})", flush=True)
    if save_latest:
        atomic_save({"adapters": lora_state_dict(model), "optimizer": optimizer.state_dict(),
                     "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                     "global_step": global_step, "best_val": best_val, "baseline_val": baseline_val,
                     "val_loss": val_loss, "config": cfg, "lora": LORA_META}, latest_path)

t_start = time.time()
last_eval_step = global_step
win_loss, win_n, n_nonfinite = 0.0, 0, 0
model.train()
for epoch in range(global_step // steps_per_epoch, NUM_EPOCHS):
    # Deterministic per-epoch order => a resumed run sees exactly the data it would have seen.
    order = torch.randperm(len(train_ds), generator=torch.Generator().manual_seed(SEED + epoch)).tolist()
    start_micro = (global_step - epoch * steps_per_epoch) * GRAD_ACCUM_STEPS
    loader = DataLoader(train_ds, batch_size=MICRO_BATCH_SIZE, sampler=order[start_micro * MICRO_BATCH_SIZE:],
                        collate_fn=collate, num_workers=0, pin_memory=USE_CUDA)
    optimizer.zero_grad(set_to_none=True)
    for it, b in enumerate(loader, start=start_micro):
        group_size = min(GRAD_ACCUM_STEPS, n_micro - (it // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS)
        x = b["input_ids"].to(DEVICE, non_blocking=True)
        y = b["labels"].to(DEVICE, non_blocking=True)
        with amp_ctx():
            loss, _ = lm_loss(model, x, y)
        if torch.isfinite(loss):
            scaler.scale(loss / group_size).backward()
            win_loss += float(loss.detach().item()); win_n += 1
        else:
            n_nonfinite += 1

        if (it + 1) % GRAD_ACCUM_STEPS == 0 or (it + 1) == n_micro:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(train_params, MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step(); global_step += 1

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t_start
                progress = min(1.0, global_step / max(1, total_steps))
                eta_sec = elapsed / max(progress, 1e-8) * (1.0 - progress)
                from datetime import datetime, timedelta
                eta_local = (datetime.now().astimezone() + timedelta(seconds=eta_sec)).strftime("%Y-%m-%d %H:%M:%S %Z")
                rec = {"epoch": epoch + 1, "step": global_step, "total_steps": total_steps,
                       "train_loss": win_loss / max(1, win_n),
                       "grad_norm": float(grad_norm), "lr": scheduler.get_last_lr()[0],
                       "nonfinite_batches": n_nonfinite, "elapsed_min": elapsed / 60,
                       "eta_min": eta_sec / 60, "expected_finish_local": eta_local}
                if USE_CUDA:
                    rec["max_mem_gb"] = torch.cuda.max_memory_allocated() / 2**30
                log_event(rec); print(rec, flush=True)
                win_loss, win_n = 0.0, 0
            do_eval = global_step % EVAL_EVERY == 0
            do_save = global_step % SAVE_EVERY == 0
            if do_eval or do_save:
                v = evaluate_loss() if do_eval else None
                if do_eval:
                    last_eval_step = global_step
                    log_event({"event": "eval", "step": global_step, "val_loss": v})
                    print(f"step={global_step} val_loss={v:.4f} (best={min(best_val, v):.4f})", flush=True)
                checkpoint(v, save_latest=True)
                model.train()

    if last_eval_step != global_step:       # end-of-epoch evaluation (skipped if we just evaluated)
        v = evaluate_loss(); last_eval_step = global_step
        log_event({"event": "epoch_end", "epoch": epoch + 1, "step": global_step, "val_loss": v})
        checkpoint(v, save_latest=True)
        elapsed = time.time() - t_start
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS} complete | step={global_step} | "
              f"val_loss={v:.4f} | best={best_val:.4f} | elapsed={elapsed/60:.1f} min", flush=True)
        model.train()
    else:
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS} complete | step={global_step} | best={best_val:.4f}")

if not best_path.exists():          # e.g. nothing improved because every evaluation was non-finite
    checkpoint(evaluate_loss(), save_latest=True)
training_summary = {"baseline_val_loss": baseline_val, "best_val_loss": float(best_val),
                    "global_step": global_step, "train_minutes": (time.time() - t_start) / 60}
print("Training complete:", training_summary, "\nBest adapters:", best_path)

Baseline validation loss (frozen Phase 1 model): 1.6739
{'epoch': 1, 'step': 50, 'total_steps': 2505, 'train_loss': 1.565617173910141, 'grad_norm': 0.2737902104854584, 'lr': 8.16e-05, 'nonfinite_batches': 0, 'elapsed_min': 1.5033667922019958, 'eta_min': 73.815309497118, 'expected_finish_local': '2026-11-01 08:12:04 UTC', 'max_mem_gb': 2.4189000129699707}
{'epoch': 1, 'step': 100, 'total_steps': 2505, 'train_loss': 1.4371611531078816, 'grad_norm': 0.20961904525756836, 'lr': 0.00016160000000000002, 'nonfinite_batches': 0, 'elapsed_min': 2.998847500483195, 'eta_min': 72.12228238662084, 'expected_finish_local': '2026-11-01 08:11:52 UTC', 'max_mem_gb': 2.4189000129699707}
{'epoch': 1, 'step': 150, 'total_steps': 2505, 'train_loss': 1.393092399984598, 'grad_norm': 0.25626736879348755, 'lr': 0.00019995099967618995, 'nonfinite_batches': 0, 'elapsed_min': 4.49590844710668, 'eta_min': 70.58576261957487, 'expected_finish_local': '2026-11-01 08:11:50 UTC', 'max_mem_gb': 2.4189000129699707}
{'epoch

In [14]:
# 11. Restore the best adapters
best_obj = load_torch(best_path)
if best_obj["lora"] != LORA_META:
    raise RuntimeError("Best checkpoint was trained with different LoRA settings than the current configuration.")
load_lora_state_dict(model, best_obj["adapters"])
model.to(DEVICE).eval()
print(f"Restored best adapters | step={best_obj['global_step']} | val_loss={best_obj['val_loss']:.4f}")
del best_obj

Restored best adapters | step=2505 | val_loss=1.2971


In [15]:
# 12. Generation (greedy, KV-cached) and validation metrics
BLOCKED_IDS = [PAD_ID, tokenizer.unk_token_id, BOS_ID]

def build_inference_prompt(document, max_new_tokens):
    # Same prompt layout as training; the source window shrinks (never the prompt) to leave room to generate.
    budget = min(MAX_SOURCE_TOKENS, MAX_SEQ_LEN - PREFIX_OVERHEAD - max_new_tokens)
    if budget < 1:
        raise ValueError("max_new_tokens leaves no room for the source document")
    src = select_source_ids(document)[0]   # same selector as training
    return build_prompt_ids(src, budget)

@torch.inference_mode()
def generate_summary(document, max_new_tokens=GEN_MAX_NEW_TOKENS, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE):
    model.eval()
    prompt = build_inference_prompt(document, max_new_tokens)
    max_new = min(max_new_tokens, model.max_positions - len(prompt))
    emb_w = model.token_emb.weight
    with amp_ctx():
        h, past = model.forward_hidden(torch.tensor([prompt], device=DEVICE), use_cache=True)
    generated, seen = [], {}
    n = no_repeat_ngram_size
    for _ in range(max_new):
        with amp_ctx():
            logits = F.linear(h[:, -1], emb_w)
        logits = logits.float()
        logits[:, BLOCKED_IDS] = -float("inf")
        key = None
        if n > 0 and len(generated) >= n - 1:
            key = tuple(generated[len(generated) - (n - 1):])     # the previous n-1 tokens
            banned = seen.get(key)
            if banned:
                logits[0, list(banned)] = -float("inf")
        nxt = int(torch.argmax(logits, dim=-1).item())
        if nxt == EOS_ID:
            break
        if key is not None:
            seen.setdefault(key, set()).add(nxt)
        generated.append(nxt)
        with amp_ctx():
            h, past = model.forward_hidden(torch.tensor([[nxt]], device=DEVICE), past_kvs=past, use_cache=True)
    return tokenizer.decode(generated).strip()

def generate_many(rows, label, every=10):
    preds, t0 = [], time.time()
    for i, r in enumerate(rows):
        preds.append(generate_summary(r["document"]))
        if (i + 1) % every == 0 or (i + 1) == len(rows):
            rate = (time.time() - t0) / (i + 1)
            print(f"[{label}] {i + 1}/{len(rows)} | {rate:.1f}s/doc | ETA {rate * (len(rows) - i - 1) / 60:.1f} min", flush=True)
    return preds

def compute_metrics(preds, refs, use_bertscore=True):
    metrics = {}
    try:
        ensure_package("rouge_score", "rouge-score")
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
        for k in ("rouge1", "rouge2", "rougeL"):
            metrics[f"{k}_f1"] = float(np.mean([s[k].fmeasure for s in scores]))
    except Exception as e:
        print("ROUGE skipped:", repr(e))
    if use_bertscore:
        try:
            ensure_package("bert_score", "bert-score")
            from bert_score import score as bert_score
            gc.collect()
            if USE_CUDA:
                torch.cuda.empty_cache()
            safe_preds = [p if p.strip() else "." for p in preds]      # empty strings break BERTScore
            _, _, f1 = bert_score(safe_preds, refs, lang="en", batch_size=BERTSCORE_BATCH_SIZE,
                                  device=str(DEVICE), verbose=False)
            metrics["bertscore_f1"] = float(f1.mean())
        except Exception as e:
            print("BERTScore skipped (needs the scoring model: internet or a cached copy):", repr(e))
    return metrics


In [16]:
# 12b. Merge LoRA into the base weights, batched KV-cached generation, decoding-grid tuning, final validation
@torch.no_grad()
def merge_lora_inplace(module):
    # W' = W0 + (alpha/r) * B A  -> removes all adapter overhead at inference time (irreversible).
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, LoRALinear):
            assert child.base.bias is None
            W = child.base.weight.data.float()
            delta = child.scaling * (child.lora_B.data.float() @ child.lora_A.data.float())
            lin = nn.Linear(child.base.in_features, child.base.out_features, bias=False).to(child.base.weight.device)
            lin.weight.data.copy_((W + delta).to(child.base.weight.dtype))
            lin.requires_grad_(False)
            setattr(module, name, lin); n += 1
        else:
            n += merge_lora_inplace(child)
    return n

model.eval()
_probe = torch.tensor([build_inference_prompt(val_rows[0]["document"], GEN_MAX_NEW_TOKENS)[:300]], device=DEVICE)
with torch.no_grad(), amp_ctx():
    _before = model(_probe)[:, -1, :].float()
n_merged = merge_lora_inplace(model)
with torch.no_grad(), amp_ctx():
    _after = model(_probe)[:, -1, :].float()
print(f"merged {n_merged} LoRA modules | max |logit diff| after merge: {float((_before - _after).abs().max()):.2e}"
      f" | same argmax: {int(_before.argmax()) == int(_after.argmax())}")

def forward_cached(model, input_ids, position_ids, past=None, key_valid=None, last_only=True):
    # Same maths as model.forward_hidden, but with explicit position ids (needed for left-padded batches).
    cos, sin = model.rope(position_ids)
    x = model.token_emb(input_ids)
    new_past = []
    for i, blk in enumerate(model.blocks):
        a, kv = blk.attn(blk.norm1(x), cos, sin, past_kv=None if past is None else past[i],
                         use_cache=True, key_valid=key_valid)
        x = x + a
        x = x + blk.ff(blk.norm2(x))
        new_past.append(kv)
    h = model.final_norm(x)
    if last_only: h = h[:, -1:, :]
    return F.linear(h, model.token_emb.weight), new_past

@torch.inference_mode()
def generate_batch(prefixes, max_new=None, min_new=0, ngram=0, rep=1.0):
    max_new = max_new or GEN_MAX_NEW_TOKENS
    B = len(prefixes); L = max(len(p) for p in prefixes)
    assert L + max_new <= model.max_positions, (L, max_new, model.max_positions)
    pad, eos = PAD_ID, EOS_ID
    ids = torch.full((B, L), pad, dtype=torch.long)
    valid = torch.zeros((B, L), dtype=torch.bool)
    for i, p in enumerate(prefixes):                       # left-pad: last column = last real token
        ids[i, L - len(p):] = torch.tensor(p, dtype=torch.long); valid[i, L - len(p):] = True
    ids, valid = ids.to(DEVICE), valid.to(DEVICE)
    pos = (valid.long().cumsum(1) - 1).clamp(min=0)        # positions restart at 0 at the first real token
    with amp_ctx():
        logits, past = forward_cached(model, ids, pos, None, valid)
    logits = logits[:, -1, :].float()
    cur_pos = pos[:, -1].clone()
    seen = torch.zeros((B, logits.size(-1)), dtype=torch.bool, device=DEVICE)
    rows_idx = torch.arange(B, device=DEVICE)
    gen = [[] for _ in range(B)]; done = [False] * B; ban = [dict() for _ in range(B)]
    for step in range(max_new):
        logits[:, BLOCKED_IDS] = -float("inf")
        if step < min_new: logits[:, eos] = -float("inf")
        if rep != 1.0:
            pen = torch.where(logits > 0, logits / rep, logits * rep)
            logits = torch.where(seen, pen, logits)
        if ngram > 1:
            for i in range(B):
                if done[i] or len(gen[i]) < ngram - 1: continue
                bl = ban[i].get(tuple(gen[i][-(ngram - 1):]))
                if bl: logits[i, list(bl)] = -float("inf")
        nxt = logits.argmax(-1)
        nxt_list = nxt.tolist()
        for i in range(B):
            if done[i]: continue
            t = nxt_list[i]
            if t == eos: done[i] = True; continue
            gen[i].append(t)
            if ngram > 1 and len(gen[i]) >= ngram:
                ban[i].setdefault(tuple(gen[i][-ngram:-1]), set()).add(t)
        if all(done): break
        seen[rows_idx, nxt] = True
        feed = torch.where(torch.tensor(done, device=DEVICE), torch.full_like(nxt, pad), nxt)
        cur_pos = cur_pos + 1
        valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=DEVICE)], 1)
        with amp_ctx():
            logits, past = forward_cached(model, feed[:, None], cur_pos[:, None], past, valid)
        logits = logits[:, -1, :].float()
    return [tokenizer.decode(g).strip() for g in gen]

def generate_many_batched(prefixes, batch_size=None, desc="generating", on_batch=None, **kw):
    batch_size = batch_size or GEN_BATCH_SIZE
    order = sorted(range(len(prefixes)), key=lambda i: len(prefixes[i]))    # similar lengths -> little padding
    out = [None] * len(prefixes); t0 = time.time(); n_done = [0]
    def run(idxs):
        try:
            res = generate_batch([prefixes[i] for i in idxs], **kw)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if len(idxs) == 1: raise
            h = len(idxs) // 2; run(idxs[:h]); run(idxs[h:]); return
        for i, s in zip(idxs, res): out[i] = s
        if on_batch is not None: on_batch(idxs, res)
        n_done[0] += len(idxs)
        rate = (time.time() - t0) / n_done[0]
        print(f"[{desc}] {n_done[0]}/{len(prefixes)} | {rate:.1f}s/doc | ETA {rate * (len(prefixes) - n_done[0]) / 60:.1f} min", flush=True)
    for s in range(0, len(order), batch_size): run(order[s:s + batch_size])
    return out

def gen_kwargs(g):
    return dict(max_new=GEN_MAX_NEW_TOKENS, min_new=int(g["min_frac"] * GEN_MAX_NEW_TOKENS), ngram=g["ngram"], rep=g["rep"])

# ---- decoding-grid tuning on held-out docs disjoint from the 100 scored validation docs ----
_start = VAL_SCORE_EXAMPLES
tune_rows = val_rows[_start:_start + N_TUNE_DOCS]
if len(tune_rows) != N_TUNE_DOCS:
    raise ValueError(f"Need {N_TUNE_DOCS} disjoint decoding-tuning docs after the {VAL_SCORE_EXAMPLES} scored docs.")
selector_ids = {str(r["id"]) for r in selector_eval_rows}
decode_tune_ids = {str(r["id"]) for r in tune_rows}
assert selector_ids.isdisjoint(decode_tune_ids), "Selector-tuning and decoding-tuning documents overlap."
tune_prefix = [build_inference_prompt(r["document"], GEN_MAX_NEW_TOKENS) for r in tune_rows]
tune_refs = [r["summary"] for r in tune_rows]

_scorer = {"bert": None}
def quick_metrics(cands, refs):
    res = {}
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    z = [sc.score(r, c) for c, r in zip(cands, refs)]
    res["rouge1"] = float(np.mean([s["rouge1"].fmeasure for s in z])); res["rougeL"] = float(np.mean([s["rougeL"].fmeasure for s in z]))
    try:
        ensure_package("bert_score", "bert-score")
        from bert_score import BERTScorer
        if _scorer["bert"] is None: _scorer["bert"] = BERTScorer(lang="en", device=str(DEVICE), batch_size=BERTSCORE_BATCH_SIZE)
        _, _, f1 = _scorer["bert"].score([c if c.strip() else "." for c in cands], refs)
        res["bertscore_f1"] = float(f1.mean())
    except Exception as e:
        print("BERTScore skipped in grid:", repr(e)[:120])
    res["avg_words"] = float(np.mean([len(c.split()) for c in cands]))
    return res

grid_results = []
for g in GEN_GRID:
    t0 = time.time()
    outs = generate_many_batched(tune_prefix, desc=g["name"], **gen_kwargs(g))
    m = quick_metrics(outs, tune_refs); m.update(name=g["name"], sec=round(time.time() - t0, 1))
    grid_results.append(m)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()}, flush=True)
_use_bert = all("bertscore_f1" in m for m in grid_results)
_best = max(grid_results, key=lambda m: m["bertscore_f1"] if _use_bert else m["rougeL"])
BEST_GEN = next(g for g in GEN_GRID if g["name"] == _best["name"])
print("Selected decoding:", BEST_GEN, "| criterion:", "BERTScore F1" if _use_bert else "ROUGE-L")

# ---- final validation on the same 100 documents Exp 1 scored (identical split seed) ----
val_subset = val_rows[:min(VAL_SCORE_EXAMPLES, len(val_rows))]
score_ids = {str(r["id"]) for r in val_subset}
assert score_ids.isdisjoint(decode_tune_ids), "Final scored validation overlaps decoding-tuning documents."
assert score_ids.isdisjoint(selector_ids), "Final scored validation overlaps selector-tuning documents."
val_prefix = [build_inference_prompt(r["document"], GEN_MAX_NEW_TOKENS) for r in val_subset]
val_preds = generate_many_batched(val_prefix, desc="val", **gen_kwargs(BEST_GEN))
val_refs = [r["summary"] for r in val_subset]
val_metrics = compute_metrics(val_preds, val_refs)
val_metrics["empty_predictions"] = int(sum(not p.strip() for p in val_preds))
val_metrics["mean_pred_words"] = float(np.mean([len(p.split()) for p in val_preds]))
val_metrics["mean_ref_words"] = float(np.mean([len(p.split()) for p in val_refs]))
print(json.dumps(val_metrics, indent=2))
print("\nExample prediction:\n", val_preds[0][:600])

merged 119 LoRA modules | max |logit diff| after merge: 7.81e-03 | same argmax: True
[greedy] 8/32 | 2.3s/doc | ETA 0.9 min
[greedy] 16/32 | 2.3s/doc | ETA 0.6 min
[greedy] 24/32 | 2.3s/doc | ETA 0.3 min
[greedy] 32/32 | 2.3s/doc | ETA 0.0 min


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (4 outputs left out)

{'rouge1': 0.3439, 'rougeL': 0.2333, 'bertscore_f1': 0.7948, 'avg_words': 386.0312, 'name': 'greedy', 'sec': 111.7}
[greedy+min] 8/32 | 2.2s/doc | ETA 0.9 min
[greedy+min] 16/32 | 2.2s/doc | ETA 0.6 min
[greedy+min] 24/32 | 2.2s/doc | ETA 0.3 min
[greedy+min] 32/32 | 2.2s/doc | ETA 0.0 min
{'rouge1': 0.3439, 'rougeL': 0.2333, 'bertscore_f1': 0.7948, 'avg_words': 386.0312, 'name': 'greedy+min', 'sec': 77.5}
[ng12+min] 8/32 | 2.2s/doc | ETA 0.9 min
[ng12+min] 16/32 | 2.2s/doc | ETA 0.6 min
[ng12+min] 24/32 | 2.2s/doc | ETA 0.3 min
[ng12+min] 32/32 | 2.2s/doc | ETA 0.0 min
{'rouge1': 0.4465, 'rougeL': 0.25, 'bertscore_f1': 0.8333, 'avg_words': 373.8438, 'name': 'ng12+min', 'sec': 77.1}
[ng20+min] 8/32 | 2.2s/doc | ETA 0.9 min
[ng20+min] 16/32 | 2.2s/doc | ETA 0.6 min
[ng20+min] 24/32 | 2.2s/doc | ETA 0.3 min
[ng20+min] 32/32 | 2.2s/doc | ETA 0.0 min
{'rouge1': 0.4206, 'rougeL': 0.2444, 'bertscore_f1': 0.8262, 'avg_words': 376.2188, 'name': 'ng20+min', 'sec': 76.9}
[ng20+rep1.05+min] 8/32 

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{
  "rouge1_f1": 0.39095043130686763,
  "rouge2_f1": 0.16865509106301207,
  "rougeL_f1": 0.22633295547294124,
  "bertscore_f1": 0.826414942741394,
  "empty_predictions": 0,
  "mean_pred_words": 366.69,
  "mean_ref_words": 952.83
}

Example prediction:
 The appellants and respondent No. 4 were the daughters and legal representatives of Savoy Ranganna who was the plaintiff in O.S. 34 of 1950 51 instituted in the court of the District Judge, Mysore. The suit was filed in the court of the Subordinate Judge, Mysore for partition of the joint family property and for partition of the joint family. The appellants and respondent No. 4; the daughters and legal representatives were the daughters and legal representatives. The suit was decreed in favour of the appellants. The appellants and respondent No. 4, the daughters of Savoy Ranganna filed a suit 


In [17]:
# 13. Save validation predictions, metrics and experiment metadata
with open(OUTPUT_DIR / "validation_predictions.json", "w", encoding="utf-8") as f:
    json.dump([{"id": r["id"], "summary": p, "reference": r["summary"]} for r, p in zip(val_subset, val_preds)],
              f, ensure_ascii=False, indent=2)

metadata = {
    "experiment": "Phase6-Exp2-Model4-Standard-LoRA",
    "base_architecture": "efficient_gqa_rope_swiglu_decoder",
    "checkpoint": str(CHECKPOINT_PATH), "tokenizer": str(TOKENIZER_PATH),
    "lora": LORA_META, "trainable_parameters": trainable, "total_parameters": total,
    "max_source_tokens": MAX_SOURCE_TOKENS, "max_target_tokens": MAX_TARGET_TOKENS, "max_seq_len": MAX_SEQ_LEN,
    "micro_batch_size": MICRO_BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS, "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
    "amp_dtype": str(AMP_DTYPE) if USE_CUDA else "none",
    "gen_max_new_tokens": GEN_MAX_NEW_TOKENS, "selected_decoding": BEST_GEN, "decoding_grid": grid_results,
    "source_selector": SELECTOR_MODE, "selector_coverage": {k: float(np.mean(v)) for k, v in cov.items()},
    "train_examples": len(train_rows), "validation_examples": len(val_rows),
    "validation_scored_examples": len(val_subset),
    "validation_strategy": "full held-out validation set",
    "runtime_budget_hours": "3-4",
    "training": training_summary, "validation_metrics": val_metrics,
    "seed": SEED, "torch_version": torch.__version__,
}
with open(OUTPUT_DIR / "experiment_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

{
  "experiment": "Phase6-Exp2-Model4-Standard-LoRA",
  "base_architecture": "efficient_gqa_rope_swiglu_decoder",
  "checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt",
  "tokenizer": "/kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json",
  "lora": {
    "rank": 16,
    "alpha": 32,
    "dropout": 0.05,
    "targets": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "variant": "standard LoRA"
  },
  "trainable_parameters": 3307520,
  "total_parameters": 102616960,
  "max_source_tokens": 1500,
  "max_target_tokens": 800,
  "max_seq_len": 2400,
  "micro_batch_size": 1,
  "grad_accum_steps": 8,
  "epochs": 3,
  "learning_rate": 0.0002,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "amp_dtype": "torch.float16",
  "gen_max_new_tokens": 800,
  "selected_decoding": {
    "name": "ng12+min",
    "min_frac": 0.45,
    "ngram": 12,
    "rep": 1.0
  },
  "decoding_grid":

In [18]:
# 13B. Save reproducibility metadata for Experiment 2
experiment_metadata = {
    "experiment": "Phase 6 Experiment 2 — Model 4 + standard LoRA",
    "base_model": "Phase-1 Model 4 Efficient Decoder (~99.31M)",
    "method": "standard additive LoRA",
    "formula": "W' = W0 + (alpha/r) * B(A(dropout(x)))",
    "rank": LORA_RANK,
    "alpha": LORA_ALPHA,
    "scaling": LORA_ALPHA / LORA_RANK,
    "dropout": LORA_DROPOUT,
    "targets": list(LORA_TARGETS),
    "seed": SEED,
    "validation_fraction": VAL_FRACTION,
    "max_source_tokens": MAX_SOURCE_TOKENS,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "max_sequence_length": MAX_SEQ_LEN,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "validation_score_examples": VAL_SCORE_EXAMPLES,
    "selector_eval_examples": SELECTOR_EVAL_DOCS,
    "selector_eval_start": SELECTOR_EVAL_START,
    "decoding_tune_examples": N_TUNE_DOCS,
    "decoding_tune_start": VAL_SCORE_EXAMPLES,
    "train_examples": len(train_examples),
    "validation_examples": len(val_examples),
    "test_examples": len(test_records),
    "trainable_parameters": trainable,
    "total_parameters": total,
}
with open(OUTPUT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(experiment_metadata, f, indent=2)
print(json.dumps(experiment_metadata, indent=2))


{
  "experiment": "Phase 6 Experiment 2 \u2014 Model 4 + standard LoRA",
  "base_model": "Phase-1 Model 4 Efficient Decoder (~99.31M)",
  "method": "standard additive LoRA",
  "formula": "W' = W0 + (alpha/r) * B(A(dropout(x)))",
  "rank": 16,
  "alpha": 32,
  "scaling": 2.0,
  "dropout": 0.05,
  "targets": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "seed": 2026,
  "validation_fraction": 0.05,
  "max_source_tokens": 1500,
  "max_target_tokens": 800,
  "max_sequence_length": 2400,
  "epochs": 3,
  "learning_rate": 0.0002,
  "validation_score_examples": 100,
  "selector_eval_examples": 150,
  "selector_eval_start": 132,
  "decoding_tune_examples": 32,
  "decoding_tune_start": 100,
  "train_examples": 6677,
  "validation_examples": 351,
  "test_examples": 100,
  "trainable_parameters": 3307520,
  "total_parameters": 102616960
}


In [19]:
# 13A. Exact Phase 6 evaluator embedded in the notebook (no separate .py file needed)
# This reproduces the supplied evaluate_phase6.py logic and runs it directly on validation outputs.
# Dependencies: rouge-score, bert-score, torch.

ensure_package("rouge_score", "rouge-score")
ensure_package("bert_score", "bert-score")
from rouge_score import tokenizers, rouge_scorer
import bert_score

def evaluator_load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def evaluator_lcs(a, b):
    # Bit-parallel LCS implementation from the supplied evaluator.
    if len(a) > len(b):
        a, b = b, a
    masks = {}
    for i, token in enumerate(a):
        masks[token] = masks.get(token, 0) | (1 << i)
    state = 0
    for token in b:
        mask = masks.get(token, 0)
        x = mask | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")

def evaluator_read_predictions(value):
    if isinstance(value, dict):
        output = value
    elif isinstance(value, list):
        output = {}
        for record in value:
            if not isinstance(record, dict) or "id" not in record or "summary" not in record:
                raise ValueError("Each prediction item must contain id and summary.")
            if record["id"] in output:
                raise ValueError(f"Duplicate prediction ID: {record['id']}")
            output[record["id"]] = record["summary"]
    else:
        raise ValueError("Prediction must be a JSON object or list of objects.")

    for key, summary in output.items():
        if not isinstance(key, str) or not isinstance(summary, str) or not summary.strip():
            raise ValueError(f"Invalid summary for ID {key!r}.")
    return output

def evaluate_phase6_exact(truth_file, prediction_file):
    truth = evaluator_load(truth_file)
    prediction_map = evaluator_read_predictions(evaluator_load(prediction_file))

    if not isinstance(truth, dict) or not truth:
        raise ValueError("Truth must be a non-empty JSON object.")
    if set(truth) != set(prediction_map):
        raise ValueError(
            f"ID mismatch: missing={len(set(truth)-set(prediction_map))}, "
            f"extra={len(set(prediction_map)-set(truth))}"
        )

    predictions = [prediction_map[key].strip() for key in truth]
    references = [truth[key].strip() for key in truth]

    scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
    tokenizer = tokenizers.DefaultTokenizer(use_stemmer=True)
    rouge1 = rouge_l = 0.0

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1 += scores["rouge1"].fmeasure

        pred_tokens = tokenizer.tokenize(pred)
        ref_tokens = tokenizer.tokenize(ref)
        if not pred_tokens and not ref_tokens:
            rouge_l_doc = 1.0
        elif not pred_tokens or not ref_tokens:
            rouge_l_doc = 0.0
        else:
            common = evaluator_lcs(pred_tokens, ref_tokens)
            precision = common / len(pred_tokens)
            recall = common / len(ref_tokens)
            rouge_l_doc = (
                0.0 if precision + recall == 0
                else 2 * precision * recall / (precision + recall)
            )
        rouge_l += rouge_l_doc

    count = len(truth)
    rouge1 /= count
    rouge_l /= count

    device = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, bert_f1 = bert_score.score(
        predictions, references, lang="en", batch_size=8,
        verbose=False, device=device
    )
    bert_f1_mean = float(bert_f1.mean().item())

    print(f"Documents: {count}")
    print(f"BERTScore F1: {bert_f1_mean:.6f}")
    print(f"ROUGE-1: {rouge1:.6f}")
    print(f"ROUGE-L: {rouge_l:.6f}")

    return {
        "documents": count,
        "bertscore_f1": bert_f1_mean,
        "rouge1": rouge1,
        "rougeL": rouge_l,
    }

# Save files in precisely the formats expected by the original evaluator.
val_truth = {str(record["id"]): record["summary"] for record in val_subset}
truth_path = OUTPUT_DIR / "validation_truth.json"
evaluator_prediction_path = OUTPUT_DIR / "validation_predictions_for_evaluator.json"

with open(truth_path, "w", encoding="utf-8") as f:
    json.dump(val_truth, f, ensure_ascii=False, indent=2)
with open(evaluator_prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        [{"id": str(record["id"]), "summary": pred}
         for record, pred in zip(val_subset, val_preds)],
        f, ensure_ascii=False, indent=2
    )

print("Running embedded version of your evaluate_phase6.py")
official_validation_metrics = evaluate_phase6_exact(
    truth_path, evaluator_prediction_path
)


Running embedded version of your evaluate_phase6.py


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Documents: 100
BERTScore F1: 0.826415
ROUGE-1: 0.390950
ROUGE-L: 0.226333


In [20]:
# 14. Official test predictions -> [{"id": "...", "summary": "..."}]
# Resumable: every prediction is appended to a .jsonl file, so an interrupted run continues where it stopped.
FALLBACK_SUMMARY = "The judgment could not be summarized from the supplied context."

if not RUN_TEST_INFERENCE:
    print("RUN_TEST_INFERENCE=False; skipping test predictions.")
else:
    partial_path = OUTPUT_DIR / "submission_partial.jsonl"
    done = {}
    if partial_path.exists() and RESUME_TEST_PREDICTIONS:
        with open(partial_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    if isinstance(rec.get("id"), str) and isinstance(rec.get("summary"), str) and rec["summary"].strip():
                        done[rec["id"]] = rec["summary"]
        print(f"Found {len(done)} cached test predictions; continuing.")
    todo = [r for r in test_records if str(r["id"]) not in done]
    prog = {"n": 0}
    with open(partial_path, "a" if (partial_path.exists() and RESUME_TEST_PREDICTIONS) else "w", encoding="utf-8") as f:
        def _on_batch(idxs, outs):
            for i, s in zip(idxs, outs):
                rid = str(todo[i]["id"]); s = s.strip() or FALLBACK_SUMMARY
                done[rid] = s
                f.write(json.dumps({"id": rid, "summary": s}, ensure_ascii=False) + "\n")
            f.flush(); prog["n"] += len(idxs)
        if todo:
            test_prefix = [build_inference_prompt(r["document"], GEN_MAX_NEW_TOKENS) for r in todo]
            generate_many_batched(test_prefix, desc="test", on_batch=_on_batch, **gen_kwargs(BEST_GEN))

    test_preds = [{"id": str(r["id"]), "summary": done[str(r["id"])]} for r in test_records]
    assert len(test_preds) == len(test_records)
    assert len({x["id"] for x in test_preds}) == len(test_preds)
    assert all(x["summary"].strip() for x in test_preds)
    # Competition archive: prediction.json must be directly at the ZIP root.
    pred_path = OUTPUT_DIR / "prediction.json"
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(test_preds, f, ensure_ascii=False, indent=2)

    import zipfile
    zip_path = OUTPUT_DIR / "phase6_exp2_model4_lora_submission.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    # Verify archive structure, IDs, and non-empty summaries before upload.
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        assert names == ["prediction.json"], f"Unexpected ZIP contents: {names}"
        archived = json.loads(zf.read("prediction.json").decode("utf-8"))
        assert len(archived) == len(test_records)
        assert [x["id"] for x in archived] == [str(r["id"]) for r in test_records]
        assert all(isinstance(x["summary"], str) and x["summary"].strip() for x in archived)

    print("JSON:", pred_path, "| records:", len(test_preds))
    print("ZIP :", zip_path, "| contents:", names)


[test] 8/100 | 2.2s/doc | ETA 3.4 min
[test] 16/100 | 2.2s/doc | ETA 3.1 min
[test] 24/100 | 2.2s/doc | ETA 2.8 min
[test] 32/100 | 2.2s/doc | ETA 2.5 min
[test] 40/100 | 2.2s/doc | ETA 2.2 min
[test] 48/100 | 2.2s/doc | ETA 1.9 min
[test] 56/100 | 2.2s/doc | ETA 1.6 min
[test] 64/100 | 2.2s/doc | ETA 1.3 min
[test] 72/100 | 2.2s/doc | ETA 1.0 min
[test] 80/100 | 2.2s/doc | ETA 0.7 min
[test] 88/100 | 2.2s/doc | ETA 0.4 min
[test] 96/100 | 2.2s/doc | ETA 0.1 min
[test] 100/100 | 2.3s/doc | ETA 0.0 min
JSON: /kaggle/working/phase6_exp2_model4_lora/prediction.json | records: 100
ZIP : /kaggle/working/phase6_exp2_model4_lora/phase6_exp2_model4_lora_submission.zip | contents: ['prediction.json']


## Experiment 2 notes / limitations

- This is **standard additive LoRA**, not LoRMA or rsLoRA. Its transformation is $W_0x + \tfrac{\alpha}{r}BAx$, which equals the frozen layer at initialisation because `B` starts at zero (the notebook asserts this numerically).
- The model is a causal decoder. Training is teacher-forced next-token prediction with loss only on the summary and EOS.
- The learned sentence selector packs high-value sentences into a bounded source window; if the target is long, the selected source is further cropped from its beginning and end. This is not a claim that the entire judgment is consumed. When prompt + source + target would exceed `MAX_SEQ_LEN`, the *source* window shrinks; the instruction and the `SUMMARY:` separator are never cut.
- Validation documents are grouped by normalised document hash to reduce duplicate leakage. For final reporting record: the split seed, number of duplicate groups, baseline and best validation loss, ROUGE-1/2/L F1 and BERTScore F1 (all are written to `experiment_metadata.json`).
- BERTScore truncates long inputs at the scoring model's own length limit (512 tokens), so very long summaries are only partly scored.
- Adapter checkpoints are small (adapters only). To reuse them later, rebuild the base model from the Phase 1 checkpoint, call `inject_lora`, then `load_lora_state_dict`.
- If GPU memory is tight: lower `MAX_SOURCE_TOKENS` to 1000 and `MAX_TARGET_TOKENS` to 400, or keep `MICRO_BATCH_SIZE=1` and raise `GRAD_ACCUM_STEPS`.
- The batched decoding grid compares minimum length, no-repeat n-gram size and repetition penalty on a dedicated 32-document tuning slice. Do not tune decoding on the 100 final scored validation documents.
- Do not train on the official test set or use its outputs for hyperparameter selection.
- KV-cached greedy decoding reduces repeated full-prefix computation during generation.
- Training computes the LM head only at supervised target positions; a sanity check compares this masked loss against standard full-logit loss.
- Fresh training runs clear stale adapter checkpoints, preventing an old best checkpoint from being mistaken for the current run.
- The final test cell writes the competition-required `prediction.json` and a ZIP with that file directly at its root, then validates IDs and non-empty summaries.
- The complete supplied `evaluate_phase6.py` scoring logic is embedded in cell 13A and is executed directly on the held-out validation predictions; no external evaluator file or `EVALUATOR_PATH` setting is needed.

- Checkpoint config loading now recognizes common RMSNorm epsilon aliases (`rms_norm_eps`, `norm_eps`, `layer_norm_epsilon`, `eps`). If absent, it warns and uses `1e-5`, the RMSNorm class default; verify this matches the original Phase 1 model if its implementation used a custom epsilon.\n

- Validation separation: selector choice uses `val_rows[132:282]`; decoding choice uses `val_rows[100:132]`; final reported metrics use `val_rows[:100]`. These slices are disjoint to avoid tuning on the examples used for the final local score.
- The sentence selector is fitted only on a reproducible sample of official training documents; no external corpus or pretrained model is used.


## Experiment 2 — run notes

- This is conventional additive LoRA, not LoRMA/LoRMA+ and not rsLoRA.
- The frozen Phase-1 Model 4 weights are the starting point.
- Keep `SEED`, `VAL_FRACTION`, data preprocessing, prompt and token limits unchanged when comparing with Experiment 1.
- The run uses three epochs and full 100-document validation scoring. The best adapter checkpoint is selected by validation loss; the final epoch is not automatically assumed to be best.
- Test summaries have no public gold references. Test inference only creates the submission file; Codabench computes the hidden-reference score.
- No KD, teacher logits, or teacher export is included.

- The selector fit adds CPU preprocessing before tokenisation. Batched generation is expected to be faster than one-document-at-a-time decoding, but the actual T4 throughput must be measured.
- Total run target: approximately 3–4 hours on a Kaggle T4, including training, full validation generation/evaluation and 100 test predictions. Actual time varies with output lengths and downloads.

- Validation tuning is split into disjoint slices: 150 docs for source-selector choice, 32 docs for decoding choice, and 100 docs for final reported metrics.
